## MERGE INTO — The Complete Guide

`MERGE INTO` is a powerful SQL command that combines **INSERT**, **UPDATE**, and **DELETE** into a single atomic operation. It compares a **target** table against a **source** table using a join condition and then takes different actions depending on whether rows match or not.

### Syntax
```sql
MERGE INTO target_table [AS target_alias]
USING source_table [AS source_alias]
ON <join_condition>
WHEN MATCHED [AND <condition>] THEN
  UPDATE SET col1 = val1, col2 = val2, ...
  -- or DELETE
WHEN NOT MATCHED [AND <condition>] THEN
  INSERT (col1, col2, ...) VALUES (val1, val2, ...)
WHEN NOT MATCHED BY SOURCE [AND <condition>] THEN
  UPDATE SET ... | DELETE
```

### Key Clauses
| Clause | Fires When | Allowed Actions |
|---|---|---|
| `WHEN MATCHED` | Source row **matches** a target row | `UPDATE SET ...` or `DELETE` |
| `WHEN NOT MATCHED` (by target) | Source row has **no match** in target | `INSERT (...)` |
| `WHEN NOT MATCHED BY SOURCE` | Target row has **no match** in source | `UPDATE SET ...` or `DELETE` |

### Common Use Cases
* **Upsert (Update + Insert)** — Sync incoming data: update existing rows, insert new ones
* **SCD Type 1** — Overwrite old values with the latest
* **Conditional deletes** — Remove rows that meet a certain condition during the merge
* **Full sync** — Delete target rows no longer present in source

Let's simulate each scenario step by step! 👇

In [0]:
%sql
-- Target table: current product inventory (must be a Delta table for MERGE)
CREATE OR REPLACE TABLE merge_demo_products (
  product_id INT,
  name STRING,
  price DOUBLE,
  stock INT,
  status STRING
) USING DELTA;

INSERT INTO merge_demo_products VALUES
  (1, 'Laptop',     999.99,  50, 'active'),
  (2, 'Mouse',       29.99, 200, 'active'),
  (3, 'Keyboard',    79.99, 150, 'active'),
  (4, 'Monitor',    349.99,  30, 'active'),
  (5, 'Webcam',      59.99,  80, 'active');

SELECT * FROM merge_demo_products ORDER BY product_id;

In [0]:
%sql
-- Source: incoming product updates from the supplier
-- product_id 2  -> price drop (29.99 -> 24.99) + restock
-- product_id 4  -> discontinued (stock = 0)
-- product_id 6  -> brand new product
-- product_id 7  -> brand new product
CREATE OR REPLACE TEMPORARY VIEW product_updates AS
SELECT * FROM VALUES
  (2, 'Mouse',       24.99, 300, 'active'),
  (4, 'Monitor',    349.99,   0, 'discontinued'),
  (6, 'Headphones', 149.99, 100, 'active'),
  (7, 'USB Hub',     39.99, 250, 'active')
AS product_updates(product_id, name, price, stock, status);

SELECT * FROM product_updates ORDER BY product_id;

### Scenario 1: Basic UPSERT (Update + Insert)
The most common pattern. For each source row:
* If the `product_id` already exists in the target → **update** it
* If the `product_id` is new → **insert** it

Expected outcome:
* product_id **2** (Mouse): price 29.99→24.99, stock 200→300
* product_id **4** (Monitor): stock 30→0, status active→discontinued
* product_id **6** (Headphones): **inserted** as new row
* product_id **7** (USB Hub): **inserted** as new row
* product_id 1, 3, 5: **unchanged**

In [0]:
%sql
MERGE INTO merge_demo_products AS target
USING product_updates AS source
ON target.product_id = source.product_id

-- Row exists in both target and source → update it
WHEN MATCHED THEN
  UPDATE SET
    target.name   = source.name,
    target.price  = source.price,
    target.stock  = source.stock,
    target.status = source.status

-- Row exists in source but NOT in target → insert it
WHEN NOT MATCHED THEN
  INSERT (product_id, name, price, stock, status)
  VALUES (source.product_id, source.name, source.price, source.stock, source.status);

In [0]:
%sql
-- Check the result: 7 rows total (5 original + 2 new)
SELECT * FROM merge_demo_products ORDER BY product_id;

### Scenario 2: MERGE with Conditional DELETE
Sometimes you want to **delete** matched rows instead of updating them based on a condition.

Here we:
* **DELETE** rows where the source says `status = 'discontinued'`
* **UPDATE** all other matched rows normally
* **INSERT** new rows as before

> **Note:** When you have multiple `WHEN MATCHED` clauses, the one with the extra `AND` condition must come **first**.

In [0]:
%sql
-- Reset the target table back to original state
TRUNCATE TABLE merge_demo_products;

INSERT INTO merge_demo_products VALUES
  (1, 'Laptop',     999.99,  50, 'active'),
  (2, 'Mouse',       29.99, 200, 'active'),
  (3, 'Keyboard',    79.99, 150, 'active'),
  (4, 'Monitor',    349.99,  30, 'active'),
  (5, 'Webcam',      59.99,  80, 'active');

SELECT '✅ Target table reset' AS status;

In [0]:
%sql
MERGE INTO merge_demo_products AS target
USING product_updates AS source
ON target.product_id = source.product_id

-- Condition with AND must come FIRST among WHEN MATCHED clauses
WHEN MATCHED AND source.status = 'discontinued' THEN
  DELETE

-- All other matches → update normally
WHEN MATCHED THEN
  UPDATE SET
    target.name   = source.name,
    target.price  = source.price,
    target.stock  = source.stock,
    target.status = source.status

WHEN NOT MATCHED THEN
  INSERT (product_id, name, price, stock, status)
  VALUES (source.product_id, source.name, source.price, source.stock, source.status);

In [0]:
%sql
-- product_id 4 (Monitor) should be DELETED
-- product_id 2 should be updated, 6 & 7 inserted
SELECT * FROM merge_demo_products ORDER BY product_id;

### Scenario 3: WHEN NOT MATCHED BY SOURCE
This clause handles target rows that have **no corresponding row in the source**. Useful for full-sync patterns where you want to flag or remove stale target rows.

Here we:
* **UPDATE** matched rows
* **INSERT** new source rows
* **Mark as 'stale'** any target row that the source doesn't mention

In [0]:
%sql
-- Reset again
TRUNCATE TABLE merge_demo_products;

INSERT INTO merge_demo_products VALUES
  (1, 'Laptop',     999.99,  50, 'active'),
  (2, 'Mouse',       29.99, 200, 'active'),
  (3, 'Keyboard',    79.99, 150, 'active'),
  (4, 'Monitor',    349.99,  30, 'active'),
  (5, 'Webcam',      59.99,  80, 'active');

SELECT '✅ Target table reset' AS status;

In [0]:
%sql
MERGE INTO merge_demo_products AS target
USING product_updates AS source
ON target.product_id = source.product_id

WHEN MATCHED THEN
  UPDATE SET
    target.name   = source.name,
    target.price  = source.price,
    target.stock  = source.stock,
    target.status = source.status

WHEN NOT MATCHED THEN
  INSERT (product_id, name, price, stock, status)
  VALUES (source.product_id, source.name, source.price, source.stock, source.status)

-- Target rows with NO match in source → mark as stale
WHEN NOT MATCHED BY SOURCE THEN
  UPDATE SET target.status = 'stale';

In [0]:
%sql
-- product_id 1, 3, 5 should now have status = 'stale'
-- product_id 2, 4 updated; product_id 6, 7 inserted
SELECT * FROM merge_demo_products ORDER BY product_id;

### Bonus: Shorthand Syntax
When source and target have **identical schemas**, you can use the `*` shorthand:

```sql
MERGE INTO target
USING source
ON target.id = source.id
WHEN MATCHED THEN UPDATE SET *           -- updates ALL columns
WHEN NOT MATCHED THEN INSERT *            -- inserts ALL columns
WHEN NOT MATCHED BY SOURCE THEN DELETE;   -- removes stale rows
```

This is the most concise form of a full-sync MERGE.

In [0]:
%sql
-- Uncomment the line below to drop the demo table when you're done
-- DROP TABLE IF EXISTS merge_demo_products;